In [ ]:
#Extract de la carpeta damage https://zenodo.org/records/15120522

In [ ]:
# Muestra imagenes de a 5, empezando desde la que elijas. Se puede conservar
# todas, eliminar todas, o elegir individualmente cuales conservar (checkbox por imagen)

from google.colab import drive
drive.mount('/content/drive')

import os
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PIL import Image
import ipywidgets as widgets

carpeta = "/content/drive/MyDrive/PFI/DataSets/damage/damage/images/"

extensiones_validas = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

imagenes = [
    f for f in sorted(os.listdir(carpeta))
    if os.path.isfile(os.path.join(carpeta, f)) and f.lower().endswith(extensiones_validas)
]

TAMANO_LOTE = 5

# Pedir desde que imagen empezar (1 = primera)
inicio = input(f"Hay {len(imagenes)} imagenes. Desde cual queres empezar? (1 a {len(imagenes)}): ")
inicio = int(inicio) - 1 if inicio.strip().isdigit() else 0
inicio = max(0, min(inicio, len(imagenes) - 1))

indice = {"i": inicio}

out = widgets.Output()
zona_checkboxes = widgets.Output()

btn_conservar_todas = widgets.Button(description="Conservar todas", button_style="success")
btn_eliminar_todas = widgets.Button(description="Eliminar todas", button_style="danger")
btn_elegir = widgets.Button(description="Elegir individualmente", button_style="warning")
btn_confirmar_seleccion = widgets.Button(description="Confirmar seleccion", button_style="info")
btn_confirmar_seleccion.layout.display = "none"

botones_principales = widgets.HBox([btn_conservar_todas, btn_eliminar_todas, btn_elegir])

checkboxes = []


def lote_actual():
    return imagenes[indice["i"] : indice["i"] + TAMANO_LOTE]


def mostrar_lote():
    global checkboxes
    checkboxes = []
    btn_confirmar_seleccion.layout.display = "none"

    with out:
        clear_output(wait=True)
        lote = lote_actual()
        if not lote:
            print("Terminaste de revisar todas las imagenes.")
            botones_principales.close()
            return

        total = len(imagenes)
        print(f"Mostrando {indice['i']+1} a {indice['i']+len(lote)} de {total}")

        fig, axes = plt.subplots(1, len(lote), figsize=(3*len(lote), 3))
        if len(lote) == 1:
            axes = [axes]
        for ax, nombre in zip(axes, lote):
            img = Image.open(os.path.join(carpeta, nombre))
            ax.imshow(img)
            ax.set_title(nombre, fontsize=7)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    with zona_checkboxes:
        clear_output(wait=True)


def avanzar():
    indice["i"] += TAMANO_LOTE
    mostrar_lote()


def conservar_todas(b):
    avanzar()


def eliminar_todas(b):
    for nombre in lote_actual():
        os.remove(os.path.join(carpeta, nombre))
    avanzar()


def elegir_individualmente(b):
    global checkboxes
    lote = lote_actual()
    checkboxes = [widgets.Checkbox(value=True, description=nombre, indent=False) for nombre in lote]

    with zona_checkboxes:
        clear_output(wait=True)
        print("Tildadas = se conservan. Destildadas = se eliminan.")
        display(widgets.VBox(checkboxes))

    btn_confirmar_seleccion.layout.display = "inline-block"


def confirmar_seleccion(b):
    lote = lote_actual()
    for nombre, chk in zip(lote, checkboxes):
        if not chk.value:
            os.remove(os.path.join(carpeta, nombre))
    avanzar()


btn_conservar_todas.on_click(conservar_todas)
btn_eliminar_todas.on_click(eliminar_todas)
btn_elegir.on_click(elegir_individualmente)
btn_confirmar_seleccion.on_click(confirmar_seleccion)

display(botones_principales, btn_confirmar_seleccion, out, zona_checkboxes)
mostrar_lote()

In [ ]:
# Eliminar annotations huerfanos
from google.colab import drive
drive.mount('/content/drive')
import os
import xml.etree.ElementTree as ET

carpeta = "/content/drive/MyDrive/PFI/DataSets/damage/damage/"
carpeta_images = os.path.join(carpeta, "images")
carpeta_annotations = os.path.join(carpeta, "annotations")

# nombres de imagenes sin extension
imagenes = {os.path.splitext(f)[0] for f in os.listdir(carpeta_images)}

annotations = [f for f in os.listdir(carpeta_annotations) if f.lower().endswith(".xml")]

eliminados = []
for ann in annotations:
    nombre_base = os.path.splitext(ann)[0]
    if nombre_base not in imagenes:
        ruta = os.path.join(carpeta_annotations, ann)
        os.remove(ruta)
        eliminados.append(ann)

print(f"Annotations totales: {len(annotations)}")
print(f"Eliminados (sin imagen asociada): {len(eliminados)}")
print(f"Restantes: {len(annotations) - len(eliminados)}")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/PFI/DataSets 2.0/damage/damage/annotations/"

count = sum(len(files) for _, _, files in os.walk(folder_path))

print(f"Cantidad de archivos: {count}")

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET

carpeta_images = "/content/drive/MyDrive/PFI/DataSets/damage/damage/images"
carpeta_annotations = "/content/drive/MyDrive/PFI/DataSets/damage/damage/annotations"

carpeta_no_damage = "/content/drive/MyDrive/PFI/DataSets/Extract/no_damage/"
carpeta_damage = "/content/drive/MyDrive/PFI/DataSets/Extract/damage/"

os.makedirs(carpeta_no_damage, exist_ok=True)
os.makedirs(carpeta_damage, exist_ok=True)

annotations = [f for f in os.listdir(carpeta_annotations) if f.lower().endswith(".xml")]

# mapa nombre de imagen (sin ext) -> archivo real en images
imagenes_dict = {os.path.splitext(f)[0]: f for f in os.listdir(carpeta_images)}

copiadas = 0
sin_imagen = 0

for ann in annotations:
    nombre_base = os.path.splitext(ann)[0]
    if nombre_base not in imagenes_dict:
        sin_imagen += 1
        continue

    img_file = imagenes_dict[nombre_base]
    ruta_img = os.path.join(carpeta_images, img_file)

    ruta_xml = os.path.join(carpeta_annotations, ann)
    tree = ET.parse(ruta_xml)
    root = tree.getroot()

    destinos = set()

    for obj in root.findall("object"):
        name_tag = obj.find("name")
        if name_tag is None:
            continue
        name = name_tag.text
        damage = name_tag.get("damage")

        if name == "Non structural damage":
            destinos.add(carpeta_no_damage)
        else:
            carpeta_tipo = os.path.join(carpeta_damage, name, damage)
            os.makedirs(carpeta_tipo, exist_ok=True)
            destinos.add(carpeta_tipo)

    for dest in destinos:
        shutil.copy2(ruta_img, os.path.join(dest, img_file))
        copiadas += 1

print(f"Copias realizadas: {copiadas}")
print(f"Annotations sin imagen: {sin_imagen}")